In [1]:
!pip install comet_ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.3/782.3 kB 16.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.2 MB/s eta 0:00:00
  Attempting uninstall: python-box
    Found existing installation: python-box 7.3.2
    Uninstalling python-box-7.3.2:
      Successfully uninstalled python-box-7.3.2


In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os
import re
import numpy as np
from tqdm.notebook import tqdm
from sklearn.metrics import precision_recall_fscore_support
from torch.optim.lr_scheduler import ReduceLROnPlateau
import comet_ml
from comet_ml import Experiment

In [17]:
os.environ['COMET_API_KEY']='yleT5U1ZI7YXSMy0WBz6OkoSy'

experiment_name = "resnet50_lstm_v5"

In [18]:
IMAGE_DIR = '/kaggle/input/datasets/adityajn105/flickr8k/Images/'
CSV_PATH = '/kaggle/input/datasets/erykskonieczny/dataset-captions/dataset_with_negatives.csv'

df = pd.read_csv(CSV_PATH)
print(f"Załadowano {len(df)} przykładów")

Załadowano 80910 przykładów


In [19]:
unique_images = df['image'].unique()

train_imgs, val_imgs = train_test_split(unique_images, test_size=0.2, random_state=42)

train_df = df[df['image'].isin(train_imgs)].reset_index(drop=True)
val_df = df[df['image'].isin(val_imgs)].reset_index(drop=True)

print(f"Trening: {len(train_df)} wierszy, Walidacja: {len(val_df)} wierszy")

Trening: 64720 wierszy, Walidacja: 16190 wierszy


In [20]:
class ImageTextDataset(Dataset):
    def __init__(self, df, image_dir, vocab, transform=None, max_len=40):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        image_path = os.path.join(self.image_dir, row['image'])
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
            
        text = row['caption'].lower()
        text = re.sub(r"[^a-z0-9\s]", "", text)
        words = text.split()
        indices = [self.vocab.get(w, self.vocab.get('<unk>', 1)) for w in words[:self.max_len]]

        if not indices: 
            indices = [self.vocab.get('<pad>', 0)]

        if len(indices) < self.max_len:
            indices += [self.vocab.get('<pad>', 0)] * (self.max_len - len(indices))

        tokens = torch.tensor(indices, dtype=torch.long)
        
        label = torch.tensor(row['label'], dtype=torch.float32)
        
        return image, tokens, label

In [21]:
all_words = []
for caption in df['caption']:
    words = re.sub(r"[^a-z0-9\s]", "", caption.lower()).split()
    all_words.extend(words)
unique_words = len(set(all_words))
print(f"Unikalne słowa: {unique_words}")

Unikalne słowa: 8828


In [22]:
all_words = []
for caption in df['caption']:
    words = re.sub(r"[^a-z0-9\s]", "", caption.lower()).split()
    all_words.extend(words)

from collections import Counter
word_counts = Counter(all_words)

N = 10000
vocab = {'<pad>': 0, '<unk>': 1}
for i, (word, _) in enumerate(word_counts.most_common(N-2)):
    vocab[word] = i + 2

print(f"Vocab size: {len(vocab)}")
torch.save(vocab, '/kaggle/working/vocab.pt')

Vocab size: 8830


In [24]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = ImageTextDataset(train_df, IMAGE_DIR, vocab, train_transform)
val_dataset = ImageTextDataset(val_df, IMAGE_DIR, vocab, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=4, pin_memory=True, persistent_workers=True)

In [25]:
class SubmissionModel(nn.Module):
    def __init__(self):
        super(SubmissionModel, self).__init__()

        self.vocab = self._load_vocab()
        self.vocab_size = len(self.vocab) if self.vocab else 10000
        self.max_len = 40

        self.cnn = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        # self.cnn = models.resnet50(weights=None)
        for param in self.cnn.parameters():
            param.requires_grad = False

        self.cnn.fc = nn.Sequential(
            nn.Linear(self.cnn.fc.in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
            
        self.embedding = nn.Embedding(self.vocab_size, 128, padding_idx=0)
        self.embedding_dropout = nn.Dropout(0.4)

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=256,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0
        )
        self.text_proj = nn.Sequential(
            nn.Linear(256 * 2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def _load_vocab(self):
        # current_dir = os.path.dirname(os.path.abspath(__file__))
        current_dir = '/kaggle/working'
        vocab_path = os.path.join(current_dir, 'vocab.pt')
        try:
            return torch.load(vocab_path, weights_only=False)
        except FileNotFoundError:
            print(f"WARNING: File {vocab_path} not found. Bad things happened!")
            return {'<pad>': 0, '<unk>': 1}

    def forward(self, images, tokens):
        img_features = self.cnn(images)

        embedded = self.embedding(tokens)
        embedded = self.embedding_dropout(embedded)
        output, _ = self.lstm(embedded)

        text_features = output.mean(dim=1)
        text_features = self.text_proj(text_features)

        combined = img_features * text_features
        
        prob = self.classifier(combined)
        return prob.squeeze(1)

    def predict(self, image_tensor, text_string):
        self.eval()
        device = next(self.parameters()).device
        with torch.no_grad():
            if len(image_tensor.shape) == 3:
                image_batch = image_tensor.unsqueeze(0).to(device)
            else:
                image_batch = image_tensor.to(device)

            text = text_string.lower()
            text = re.sub(r"[^a-z0-9\s]", "", text)
            words = text.split()

            indices = [self.vocab.get(w, self.vocab.get('<unk>', 1)) for w in words[:self.max_len]]
            if not indices: 
                indices = [self.vocab.get('<pad>', 0)]
                
            if len(indices) < self.max_len:
                indices += [self.vocab.get('<pad>', 0)] * (self.max_len - len(indices))

            tokens = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)
            
            score = self.forward(image_batch, tokens)
            return score.item()

In [26]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
model = SubmissionModel().to(device)

In [27]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0005, weight_decay=1e-4)
criterion = nn.BCELoss()
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

epochs = 25
start_epoch = 0

In [28]:
exp = comet_ml.start(
    workspace="eryks1",
    project_name="dlf_image_text_matching",
    experiment_config=comet_ml.ExperimentConfig(name=experiment_name)
)

learning_parameters = {
    'batch_size': 32,
    'learning_rate': 0.0005,
    'criterion': 'BCELoss',
    'optimizer': 'Adam',
    'epochs': epochs,
    'vocab_size': len(vocab),
    'embedding_dim': 128,
    'hidden_size': 256,
    'lstm_layers': 1,
    'bidirectional': True,
    'image_model': 'ResNet50 (pretrained)',
    'tokenizer': 'simple hash-based (letters + digits)',
}

exp.log_parameters(learning_parameters)
exp.log_code()

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Couldn't find a Git repository in '/kaggle/working' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
COMET INFO: Experiment is live on comet.com https://www.comet.com/eryks1/dlf-image-text-matching/d034a6f68cd6487e9408d1b16b7c1fc8

COMET WARNING: unable to find caller source code in a jupyter notebook; ignoring


In [21]:
# checkpoint_path = '/kaggle/working/best_model.pth'

# model = SubmissionModel(vocab=vocab).to(device)

# checkpoint = torch.load(checkpoint_path, map_location=device)
# model.load_state_dict(checkpoint)

# print(f"Wczytano model z: {checkpoint_path}")

# optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0005, weight_decay=1e-4)
# scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# # checkpoint = torch.load(checkpoint_path)
# # model.load_state_dict(checkpoint['model_state_dict'])
# # optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# # scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
# # start_epoch = checkpoint['epoch'] + 1
# # best_val_acc = checkpoint['best_val_acc']
# # print(f"Wznowiono od epoki {start_epoch}")

# start_epoch = 15
# epochs = 25

# print(f"Rozpoczynam trening od epoki {start_epoch+1}")

Wczytano model z: /kaggle/working/best_model.pth
Rozpoczynam trening od epoki 16


In [29]:
train_losses = []
val_losses = []
train_accs = []
val_accs = []
best_val_acc = 0.0
patience_counter = 0

for epoch in range(start_epoch, epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0

    for images, tokens, labels in tqdm(train_loader, leave=True, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
        images = images.to(device, non_blocking=True)
        tokens = tokens.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        outputs = model(images, tokens)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_correct += ((outputs > 0.5) == labels).sum().item()

    train_loss /= len(train_loader)
    train_acc = train_correct / len(train_dataset)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_preds = []
    val_labels_list = []
    
    with torch.no_grad():
        for images, tokens, labels in tqdm(val_loader, leave=True, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
            images = images.to(device, non_blocking=True)
            tokens = tokens.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images, tokens)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            preds = (outputs > 0.5).float()
            val_correct += (preds == labels).sum().item()
            
            val_preds.extend(preds.cpu().numpy())
            val_labels_list.extend(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc = val_correct / len(val_dataset)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    precision, recall, f1, _ = precision_recall_fscore_support(val_labels_list, val_preds, average='binary', zero_division=0)

    exp.log_metrics({
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'val_f1': f1,
        'val_precision': precision,
        'val_recall': recall,
        'epoch': epoch + 1,
        'learning_rate': optimizer.param_groups[0]['lr'],
        'best_val_acc': best_val_acc,
    }, epoch=epoch)
    
    print(f"Epoch {epoch+1}/{epochs}: "
          f"Train Loss {train_loss:.4f} | Train Acc {train_acc:.4f} | "
          f"Val Loss {val_loss:.4f} | Val Acc {val_acc:.4f} | "
          f"Val F1 {f1:.4f} | Precision {precision:.4f} | Recall {recall:.4f} | LR {optimizer.param_groups[0]['lr']}")

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f'best_model.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_acc': best_val_acc,
        }, 'checkpoint_full.pth')
        print(f"Nowy najlepszy model zapisany (val acc: {val_acc:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1

    torch.cuda.empty_cache()

Epoch 1/25 [Train]:   0%|          | 0/2023 [00:00<?, ?it/s]

Epoch 1/25 [Val]:   0%|          | 0/506 [00:00<?, ?it/s]

Epoch 1/25: Train Loss 0.5892 | Train Acc 0.6726 | Val Loss 0.5058 | Val Acc 0.7432 | Val F1 0.7861 | Precision 0.6735 | Recall 0.9440 | LR 0.0005
Nowy najlepszy model zapisany (val acc: 0.7432)


Epoch 2/25 [Train]:   0%|          | 0/2023 [00:00<?, ?it/s]

Epoch 2/25 [Val]:   0%|          | 0/506 [00:00<?, ?it/s]

Epoch 2/25: Train Loss 0.4880 | Train Acc 0.7617 | Val Loss 0.4415 | Val Acc 0.7929 | Val F1 0.8136 | Precision 0.7395 | Recall 0.9043 | LR 0.0005
Nowy najlepszy model zapisany (val acc: 0.7929)


... (65 outputs left out)

Epoch 24/25: Train Loss 0.2197 | Train Acc 0.9114 | Val Loss 0.3280 | Val Acc 0.8707 | Val F1 0.8736 | Precision 0.8545 | Recall 0.8936 | LR 0.0005
Nowy najlepszy model zapisany (val acc: 0.8707)


Epoch 25/25 [Train]:   0%|          | 0/2023 [00:00<?, ?it/s]

Epoch 25/25 [Val]:   0%|          | 0/506 [00:00<?, ?it/s]

Epoch 25/25: Train Loss 0.2177 | Train Acc 0.9114 | Val Loss 0.3358 | Val Acc 0.8588 | Val F1 0.8559 | Precision 0.8740 | Recall 0.8385 | LR 0.0005


In [ ]:
torch.save(model.state_dict(), 'weights.pth')
print("Weights saved in file weights.pth")

In [30]:
exp.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : resnet50_lstm_v5
COMET INFO:     url                   : https://www.comet.com/eryks1/dlf-image-text-matching/d034a6f68cd6487e9408d1b16b7c1fc8
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     best_val_acc [25]  : (0.0, 0.870722668313774)
COMET INFO:     epoch [25]         : (1, 25)
COMET INFO:     learning_rate      : 0.0005
COMET INFO:     train_acc [25]     : (0.6725741656365883, 0.911433868974042)
COMET INFO:     train_loss [25]    : (0.21767112897875665, 0.5892367907903363)
COMET INFO:     val_acc [25]       : (0.7431747992588017, 0.870722668313774)
COMET INFO:     val_f1 [25]        : (0.7861331138771731, 0.8736860649480298)
COMET INFO: